In [ ]:
import numpy as np
import pandas as pd

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Sheet1"
series_col = "y"  # 原始序列列名

df = pd.read_excel(file_path, sheet_name=sheet_name)
x0 = df[series_col].to_numpy(dtype=float)

# ========= 2) 参数模板 =========
params = {
    "n_future": 5  # int: 向后预测期数
}

def gm11_predict(x0, n_future):
    """GM(1,1) 灰色预测"""
    x0 = np.array(x0, dtype=float)
    n = len(x0)
    x1 = np.cumsum(x0)
    z1 = 0.5 * (x1[:-1] + x1[1:])
    B = np.column_stack((-z1, np.ones(n - 1)))
    Y = x0[1:].reshape(-1, 1)
    a, b = np.linalg.lstsq(B, Y, rcond=None)[0].flatten()

    def x1_hat(k):
        return (x0[0] - b / a) * np.exp(-a * (k - 1)) + b / a

    x_hat = []
    for k in range(1, n + n_future + 1):
        if k == 1:
            x_hat.append(x0[0])
        else:
            x_hat.append(x1_hat(k) - x1_hat(k - 1))
    return np.array(x_hat)

pred = gm11_predict(x0, params["n_future"])
print(pred)

In [ ]:
"""
灰色预测模型

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "灰色预测模型.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
TARGET_COLUMN = "y"  # TODO: 请填写[目标列名]，说明：必须为正数时间序列。
FORECAST_STEPS = 3  # TODO: 请填写[预测期数]，说明：正整数，不宜过长。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    # GM(1,1) 只需要一列非负时间序列，序列应尽量呈单调趋势。
    x0 = data[TARGET_COLUMN].to_numpy(dtype=float)
    if np.any(x0 <= 0):
        raise ValueError("GM(1,1) 要求原始序列为正数，请先平移或检查数据。")

    x1 = np.cumsum(x0)
    z1 = 0.5 * (x1[1:] + x1[:-1])
    B = np.column_stack((-z1, np.ones(len(z1))))
    Y = x0[1:]
    a, b = np.linalg.lstsq(B, Y, rcond=None)[0]

    def x1_hat(k: int) -> float:
        return (x0[0] - b / a) * np.exp(-a * k) + b / a

    fitted = [x0[0]] + [x1_hat(k) - x1_hat(k - 1) for k in range(1, len(x0))]
    future = [x1_hat(k) - x1_hat(k - 1) for k in range(len(x0), len(x0) + FORECAST_STEPS)]
    result = pd.DataFrame({"序号": range(1, len(x0) + FORECAST_STEPS + 1), "拟合或预测值": fitted + future})
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result)


if __name__ == "__main__":
    df = load_data()
    run_model(df)
